In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community
!pip install -q sentence-transformers faiss-cpu
!pip install -q pypdf


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 23.4 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

print("Import successful")

/tmp/ipykernel_1045/4150264707.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Import successful


In [ ]:
!pip show langchain
!pip show langchain-community

Name: langchain
Version: 1.3.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: 


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

print("All imports successful")

All imports successful


In [ ]:
import requests

url = "https://arxiv.org/pdf/1706.03762.pdf"

pdf_data = requests.get(url).content

with open("attention_is_all_you_need.pdf", "wb") as f:
    f.write(pdf_data)

print("PDF downloaded")

PDF downloaded


In [ ]:
loader = PyPDFLoader(
    "attention_is_all_you_need.pdf"
)

documents = loader.load()

print("Pages:", len(documents))

Pages: 15


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(
    documents
)

print("Chunks:", len(chunks))

Chunks: 103


In [ ]:
embed_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

texts = [chunk.page_content for chunk in chunks]

embeddings = embed_model.encode(
    texts
)

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(103, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

print("FAISS index ready")

FAISS index ready


In [ ]:
def retrieve_context(query, k=3):

    query_embedding = embed_model.encode(
        [query]
    )

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    retrieved = []

    for idx in indices[0]:
        retrieved.append(
            texts[idx]
        )

    return "\n\n".join(retrieved)

In [ ]:
query = "What is the Transformer architecture?"

context = retrieve_context(query)

print(context[:2000])

Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].

To the best of our knowledge, however, the Transformer is the first transduction 

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("Mistral loaded")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Mistral loaded


In [ ]:
def generate_text(prompt):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
def answer_question(question):

    context = retrieve_context(
        question
    )

    prompt = f"""
Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    response = generate_text(
        prompt
    )

    return response

In [ ]:
answer = answer_question(
    "What is the Transformer architecture?"
)

print(answer)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Answer the question using ONLY the provided context.

Context:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].

To the best of ou

In [ ]:
evaluation_dataset = [

    {
        "question":
        "What is the Transformer architecture?",

        "expected_keywords":
        [
            "self-attention",
            "encoder",
            "decoder"
        ]
    },

    {
        "question":
        "What replaces recurrence in the Transformer?",

        "expected_keywords":
        [
            "attention"
        ]
    },

    {
        "question":
        "How many layers are used in the encoder and decoder?",

        "expected_keywords":
        [
            "6"
        ]
    }
]

In [ ]:
def keyword_accuracy(
    answer,
    expected_keywords
):

    answer = answer.lower()

    hits = 0

    for keyword in expected_keywords:

        if keyword.lower() in answer:
            hits += 1

    return hits / len(expected_keywords)

In [ ]:
question = evaluation_dataset[0]["question"]

answer = answer_question(question)

print(answer)

score = keyword_accuracy(
    answer,
    evaluation_dataset[0]["expected_keywords"]
)

print("Score:", score)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Answer the question using ONLY the provided context.

Context:
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-

The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].

To the best of ou

In [ ]:
def retrieval_hit(
    question,
    expected_keywords
):

    context = retrieve_context(question)

    context = context.lower()

    hits = 0

    for keyword in expected_keywords:

        if keyword.lower() in context:
            hits += 1

    return hits / len(expected_keywords)

In [ ]:
question = evaluation_dataset[0]["question"]

score = retrieval_hit(
    question,
    evaluation_dataset[0]["expected_keywords"]
)

print("Retrieval Score:", score)

Retrieval Score: 1.0


In [ ]:
def hallucination_score(answer, context):

    answer_words = set(
        answer.lower().split()
    )

    context_words = set(
        context.lower().split()
    )

    overlap = answer_words.intersection(
        context_words
    )

    if len(answer_words) == 0:
        return 0

    return len(overlap) / len(answer_words)

In [ ]:
question = evaluation_dataset[0]["question"]

context = retrieve_context(question)

answer = answer_question(question)

score = hallucination_score(
    answer,
    context
)

print("Groundedness Score:", score)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Groundedness Score: 0.8375


In [ ]:
def evaluate_question(question,
                      expected_keywords):

    context = retrieve_context(
        question
    )

    answer = answer_question(
        question
    )

    answer_score = keyword_accuracy(
        answer,
        expected_keywords
    )

    retrieval_score = retrieval_hit(
        question,
        expected_keywords
    )

    groundedness = hallucination_score(
        answer,
        context
    )

    return {
        "question": question,
        "answer_score": answer_score,
        "retrieval_score": retrieval_score,
        "groundedness": groundedness
    }

In [ ]:
def evaluate_dataset():

    results = []

    for item in evaluation_dataset:

        result = evaluate_question(
            item["question"],
            item["expected_keywords"]
        )

        results.append(result)

    return results

In [ ]:
def generate_evaluation_report():

    results = evaluate_dataset()

    answer_scores = []
    retrieval_scores = []
    groundedness_scores = []

    for r in results:

        answer_scores.append(
            r["answer_score"]
        )

        retrieval_scores.append(
            r["retrieval_score"]
        )

        groundedness_scores.append(
            r["groundedness"]
        )

    avg_answer = (
        sum(answer_scores)
        / len(answer_scores)
    )

    avg_retrieval = (
        sum(retrieval_scores)
        / len(retrieval_scores)
    )

    avg_groundedness = (
        sum(groundedness_scores)
        / len(groundedness_scores)
    )

    print("\n===== DocuSense Evaluation Report =====")

    print(
        f"\nAnswer Accuracy: {avg_answer:.2f}"
    )

    print(
        f"Retrieval Accuracy: {avg_retrieval:.2f}"
    )

    print(
        f"Groundedness Score: {avg_groundedness:.2f}"
    )

In [ ]:
generate_evaluation_report()

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



===== DocuSense Evaluation Report =====

Answer Accuracy: 1.00
Retrieval Accuracy: 1.00
Groundedness Score: 0.88
